# Lab 1: Trade-Weighted Spatial Lag Model — Americas

This notebook implements Lab 1 from *The New Regional Economics*.

**What you will learn:**
1. Build a trade-weighted spatial weight matrix from bilateral trade data
2. Estimate a Spatial Autoregressive (SAR) model of GDP growth
3. Interpret the spatial lag coefficient (ρ) as measuring trade-mediated spillovers
4. Run robustness specifications varying covariates and sample

**Prerequisites:** numpy, pandas, scipy (all pre-installed in Colab)

**Estimated time:** 2–3 hours

---

## 0. Setup

Clone the repository and install dependencies.

In [ ]:
# Clone the repository (only needed once per Colab session)
!git clone https://github.com/laurencehw/regional-economics.git 2>/dev/null || echo "Already cloned"
import os
os.chdir("regional-economics")

# Install dependencies
!pip install -q numpy pandas scipy requests pycountry

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add lab code to path
sys.path.insert(0, "labs/lab1_americas/code")

from lab1_americas_sar_scaffold import (
    build_trade_matrix,
    row_standardize,
    prepare_cross_section,
    estimate_sar_manual,
    run_spatial_or_fallback,
    synthetic_inputs,
)

print("Setup complete.")

## 1. Generate Synthetic Data

We start with synthetic data to understand the method, then move to real data.
The synthetic dataset has 12 regions with random trade flows and GDP characteristics.

In [ ]:
panel_df, trade_df = synthetic_inputs()

print(f"Panel: {panel_df.shape[0]} rows, {panel_df['region'].nunique()} regions")
print(f"Trade: {trade_df.shape[0]} bilateral flows")
display(panel_df.head())
display(trade_df.head())

## 2. Build the Spatial Weight Matrix

The weight matrix **W** captures the spatial relationship between regions.
In this lab, W is constructed from bilateral trade flows and row-standardized
so that each row sums to 1.

In [ ]:
# Prepare cross-section
x_cols = ["log_gdp_pc", "manufacturing_share", "border_delay_index"]
xsec = prepare_cross_section(panel_df, 2024, "region", "gdp_growth", x_cols)
regions = xsec["region"].tolist()

# Build and standardize weight matrix
w_raw = build_trade_matrix(trade_df, regions, "origin", "destination", "trade_value")
w = row_standardize(w_raw)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im0 = axes[0].imshow(w_raw, cmap="YlOrRd")
axes[0].set_title("Raw Trade Matrix")
axes[0].set_xticks(range(len(regions)))
axes[0].set_xticklabels(regions, rotation=90, fontsize=8)
axes[0].set_yticks(range(len(regions)))
axes[0].set_yticklabels(regions, fontsize=8)
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(w, cmap="YlOrRd")
axes[1].set_title("Row-Standardized W")
axes[1].set_xticks(range(len(regions)))
axes[1].set_xticklabels(regions, rotation=90, fontsize=8)
axes[1].set_yticks(range(len(regions)))
axes[1].set_yticklabels(regions, fontsize=8)
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

print(f"W density: {np.count_nonzero(w) / w.size:.2%}")
print(f"Row sums (should be ~1): {w.sum(axis=1).round(4)}")

## 3. Estimate the SAR Model

The SAR model estimates:

$$y = \rho W y + X\beta + \varepsilon$$

where $\rho$ measures the strength of spatial spillovers mediated by trade.

In [ ]:
y = xsec["gdp_growth"].to_numpy(dtype=float)
x = xsec[x_cols].to_numpy(dtype=float)

summary = run_spatial_or_fallback(y=y, x=x, w_matrix=w, x_names=x_cols)

print(f"Estimation method: {summary['method']}")
print(f"\nSpatial lag (ρ): {summary.get('rho', 'N/A')}")
print(f"\nCoefficients:")
names = summary.get('beta_names', ['intercept'] + x_cols)
for name, beta in zip(names, summary['betas']):
    print(f"  {name:30s} {beta:8.4f}")
print(f"\nN observations: {summary['n_obs']}")

## 4. Load Real Data (Americas)

Now switch to real WDI + Comtrade data for 34 American economies.
If the real data files exist in the repository, load them. Otherwise,
this section will use the pre-computed outputs.

In [ ]:
real_panel_path = Path("labs/lab1_americas/data/real_americas/panel_mapped.csv")
real_trade_path = Path("labs/lab1_americas/data/real_americas/trade_mapped.csv")

if real_panel_path.exists() and real_trade_path.exists():
    real_panel = pd.read_csv(real_panel_path)
    real_trade = pd.read_csv(real_trade_path)
    print(f"Loaded real data: {real_panel.shape[0]} panel rows, {real_trade.shape[0]} trade flows")
    print(f"Regions: {sorted(real_panel['region'].unique())}")
    print(f"Years: {sorted(real_panel['year'].unique())}")
else:
    print("Real data files not found in repository.")
    print("To use real data, run the data preparation pipeline first:")
    print("  python labs/lab1_americas/code/prepare_lab1_inputs.py --help")

## 5. Exercises

### Exercise 1: Vary the Weight Matrix
Replace trade-weighted W with a binary contiguity matrix (1 if two countries share a border, 0 otherwise). How does ρ change?

### Exercise 2: Sensitivity to Year
Re-estimate the SAR for different cross-section years. Is ρ stable over time?

### Exercise 3: Subsample Analysis
Estimate separately for USMCA members (USA, CAN, MEX) and for South American countries. How do spillover patterns differ?

### Exercise 4: Interpretation
A positive ρ means that a country's GDP growth is positively associated with the trade-weighted average growth of its partners. What economic mechanisms could produce this? What would a negative ρ imply?

In [ ]:
# Space for student work
